# FusedHexapodModel V2.7 — Clean Slate Training
4 Heads: Stereo + YOLO (35 classes) + Seg (6 depth-derived classes) + Surface Normals
Single-channel grayscale input. No teacher dependencies for Seg.

In [ ]:
from albumentations.pytorch import ToTensorV2
from contextlib import nullcontext
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import PowerNorm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from scipy.ndimage import sobel as scipy_sobel, uniform_filter
import torch, torchvision, timm
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2, os, glob, re, math, time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# =====================================================================
# V2.7 CONFIG
# =====================================================================
CONFIG = {
    'img_height': 480, 'img_width': 640,
    'num_det_classes': 40,   # ✅ V2.7: 40 robot-relevant classes   # ✅ V2.7: 35 robot-relevant classes (was 80)
    'num_seg_classes': 6,    # ✅ V2.7: depth-derived geometric classes (was 11)
    'max_disp_pixel': 192,
    'backbone_stride': 4,
    'internal_disp_steps': 48,
    'batch_size': 4,
    'ACCUMULATION_STEPS': 12,
    'lr': 2e-4,
    'num_epochs': 25,
    'num_workers': 4,
    'save_dir': "./checkpoints",
    'PHASE2_EPOCH': 25,     # Phase 2 starts after Phase 1
    # TartanAir camera params (after resize to 640x480)
    'tartan_fx': 320.0,
    'tartan_fy': 240.0,      # 320 * (480/640) — NOT 320!
    'tartan_baseline': 0.25,
    # Seg class weights (inverse frequency, tune after first epoch)
    'seg_class_weights': [1.0, 10.0, 0.5, 5.0, 5.0, 0.3],  # WALL 0.5× (suppress), STEP 10×, OBSTACLE/VEG 5×
}

SEG_CLASS_NAMES = ['WALKABLE', 'STEP', 'WALL', 'OBSTACLE', 'VEGETATION', 'VOID']

# 35 Robot-Relevant COCO Classes
ROBOT_CAT_IDS = [1,2,3,4,6,8,10,11,13,14,15,16,17,18,27,28,31,33,44,47,51,
                  62,63,64,65,67,70,72,73,75,76,77,78,79,81,82,84,85,86,88]
robot_cat_to_continuous = {cid: idx for idx, cid in enumerate(ROBOT_CAT_IDS)}

print(f"V2.7 Config: {CONFIG['num_det_classes']} det classes, {CONFIG['num_seg_classes']} seg classes")
print(f"Seg classes: {SEG_CLASS_NAMES}")


## V2.7 Architecture
1-channel input, NormalsHead, modified LRASPPHead with normals input, 35-class YOLO

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import math

# --- Hailo-8 Compatible Building Blocks (unchanged from V2.5) ---
class DWSepConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, bias=True, dilation=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, stride=stride, padding=padding,
                            dilation=dilation, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
    def forward(self, x):
        return self.pw(self.dw(x))

# --- FPN Neck (unchanged from V2.5) ---
FPN_CH = 64

class LightFPNNeck(nn.Module):
    def __init__(self, ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH):
        super().__init__()
        self.lat_s8  = nn.Conv2d(ch_s8,  fpn_ch, 1, bias=False)
        self.lat_s16 = nn.Conv2d(ch_s16, fpn_ch, 1, bias=False)
        self.lat_s32 = nn.Conv2d(ch_s32, fpn_ch, 1, bias=False)
        self.smooth_s8  = DWSepConv(fpn_ch, fpn_ch, 3, padding=1, bias=False)
        self.smooth_s16 = DWSepConv(fpn_ch, fpn_ch, 3, padding=1, bias=False)
        self.bu_s16 = DWSepConv(fpn_ch, fpn_ch, 3, stride=2, padding=1, bias=False)
        self.bu_s32 = DWSepConv(fpn_ch, fpn_ch, 3, stride=2, padding=1, bias=False)

    def forward(self, f_s8, f_s16, f_s32):
        p32 = self.lat_s32(f_s32)
        p16 = self.lat_s16(f_s16) + F.interpolate(p32, scale_factor=2, mode='nearest')
        p8  = self.lat_s8(f_s8) + F.interpolate(p16, scale_factor=2, mode='nearest')
        p8  = self.smooth_s8(p8)
        p16 = self.smooth_s16(p16) + self.bu_s16(p8)
        p32 = p32 + self.bu_s32(p16)
        return p8, p16, p32

# --- Cost Volume (unchanged) ---
class CoarseCostVolume(nn.Module):
    def __init__(self, max_disp, in_channels):
        super().__init__()
        self.max_disp = max_disp
        self.corr = nn.Conv2d(in_channels * 2, max_disp, 1, bias=True)
    def forward(self, feat_l, feat_r):
        B, C, H, W = feat_l.shape
        cost_slices = []
        for d in range(self.max_disp):
            if d == 0:
                cost_slices.append(torch.cat([feat_l, feat_r], dim=1))
            else:
                shifted = torch.zeros_like(feat_r)
                shifted[:, :, :, d:] = feat_r[:, :, :, :-d]
                cost_slices.append(torch.cat([feat_l, shifted], dim=1))
        cost = torch.stack(cost_slices, dim=2)
        B, C2, D, H, W = cost.shape
        cost = cost.permute(0, 2, 1, 3, 4).reshape(B * D, C2, H, W)
        out = self.corr(cost)
        out = out.view(B, D, self.max_disp, H, W)
        return out[:, :, 0, :, :].squeeze(2) if out.shape[2] == 1 else out.diagonal(dim1=1, dim2=2).permute(0, 3, 1, 2)

# --- Refinement Stage with Edge Guidance (from V2.5 Phase 2) ---
class RefinementStage(nn.Module):
    def __init__(self, guidance_channels, scale_factor, use_edge_guidance=False):
        super().__init__()
        self.scale_factor = scale_factor
        self.use_edge_guidance = use_edge_guidance
        extra = 1 if use_edge_guidance else 0
        self.net = nn.Sequential(
            nn.Conv2d(1 + guidance_channels + extra, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1)
        )
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def _edge_map(self, img):
        gx = F.conv2d(img, self.kx, padding=1)
        gy = F.conv2d(img, self.ky, padding=1)
        return torch.sqrt(gx**2 + gy**2 + 1e-6)

    def forward(self, disparity_low, guidance, gray_img=None):
        disparity_up = F.interpolate(
            disparity_low, scale_factor=self.scale_factor,
            mode='bilinear', align_corners=False
        ) * self.scale_factor
        inp = [disparity_up, guidance]
        if self.use_edge_guidance:
            assert gray_img is not None
            if gray_img.shape[-2:] != disparity_up.shape[-2:]:
                gray_img = F.interpolate(gray_img, size=disparity_up.shape[-2:],
                                         mode='bilinear', align_corners=False)
            inp.append(self._edge_map(gray_img))
        return F.relu(disparity_up + self.net(torch.cat(inp, dim=1)))

# --- Stereo Head with Context Network (from V2.5 Phase 2) ---
class HierarchicalStereoHead(nn.Module):
    def __init__(self, ch_s8, ch_s4, max_disp_s8):
        super().__init__()
        self.max_disp_s8 = max_disp_s8
        self.reduce_s8 = nn.Conv2d(ch_s8, 32, 1, bias=False)
        self.reduce_s4 = nn.Conv2d(ch_s4, 32, 1, bias=False)
        self.stereo_coarse = CoarseCostVolume(max_disp=self.max_disp_s8, in_channels=32)
        self.stereo_refine_s4 = RefinementStage(guidance_channels=32, scale_factor=2.0)
        self.stereo_refine_s1 = RefinementStage(guidance_channels=1, scale_factor=4.0,
                                                 use_edge_guidance=True)
        self.register_buffer('disp_reg',
            torch.arange(self.max_disp_s8, dtype=torch.float32).view(1, -1, 1, 1))
        self.temperature = 0.7
        self.context_weight = 1.0
        self.context = nn.Sequential(
            nn.Conv2d(max_disp_s8, max_disp_s8, 3, padding=1, groups=max_disp_s8, bias=False),
            nn.Conv2d(max_disp_s8, max_disp_s8, 1, bias=True),
        )

    def forward(self, l_s8, r_s8, l_s4, l_img_raw):
        feat_l_s8 = self.reduce_s8(l_s8)
        feat_r_s8 = self.reduce_s8(r_s8)
        feat_l_s4 = self.reduce_s4(l_s4)
        vol_s8 = self.stereo_coarse(feat_l_s8, feat_r_s8)
        vol_ctx = self.context(vol_s8)
        vol_s8 = self.context_weight * vol_ctx + (1.0 - self.context_weight) * vol_s8
        prob_s8 = F.softmax(vol_s8 / self.temperature, dim=1)
        disp_s8 = torch.sum(prob_s8 * self.disp_reg, dim=1, keepdim=True)
        disp_s4 = self.stereo_refine_s4(disp_s8, feat_l_s4)
        final_disp = self.stereo_refine_s1(disp_s4, l_img_raw, gray_img=l_img_raw)
        return final_disp, disp_s8

# ✅ V2.7: Normals Head — dilated convs for large receptive field
class NormalsHead(nn.Module):
    """
    Predict per-pixel surface normals from backbone features at stride 4.
    Dilated convolutions give 17×17 receptive field (= 68×68 pixels at full res)
    while keeping parameter count low (~29K, <1% of model).
    All ops Hailo-8 compatible (DWSepConv + standard Conv2d).
    """
    def __init__(self, in_ch_s4):
        super().__init__()
        self.net = nn.Sequential(
            DWSepConv(in_ch_s4, 96, 3, padding=1),       # RF: 3×3
            nn.ReLU(inplace=True),
            DWSepConv(96, 96, 3, padding=2, dilation=2),  # RF: 7×7
            nn.ReLU(inplace=True),
            DWSepConv(96, 96, 3, padding=4, dilation=4),  # RF: 15×15
            nn.ReLU(inplace=True),
            DWSepConv(96, 48, 3, padding=1),              # RF: 17×17
            nn.ReLU(inplace=True),
            nn.Conv2d(48, 3, 1),
        )
    def forward(self, feat_s4):
        raw = self.net(feat_s4)
        return F.normalize(raw, dim=1, eps=1e-6)

# ✅ V2.7: LRASPPHead with Normals input + 6 classes
class LRASPPHead(nn.Module):
    def __init__(self, low_ch, high_ch, num_classes, normals_ch=3):
        super().__init__()
        self.cbr_high = nn.Sequential(
            nn.Conv2d(high_ch, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.scale_high = nn.Sequential(
            nn.AvgPool2d(kernel_size=(30, 40)),
            nn.Conv2d(high_ch, 128, 1, bias=False),
            nn.Sigmoid()
        )
        # ✅ V2.7: low_classifier takes backbone features + predicted normals
        self.low_classifier = nn.Conv2d(low_ch + normals_ch, num_classes, 1)
        self.high_classifier = nn.Conv2d(128, num_classes, 1)
        # Dilated conv also gets normals (activated in Phase 2)
        self.mid_classifier = nn.Conv2d(128 + normals_ch, num_classes, 3, padding=2, dilation=2)
        self.use_mid = False

    def forward(self, x_low, x_high, normals_s4=None):
        out = self.cbr_high(x_high) * self.scale_high(x_high)
        out = F.interpolate(out, scale_factor=4.0, mode='bilinear', align_corners=False)
        if normals_s4 is not None:
            low_in = torch.cat([x_low, normals_s4], dim=1)
        else:
            low_in = F.pad(x_low, (0,0,0,0,0,3))  # Zero-pad if no normals
        result = self.low_classifier(low_in) + self.high_classifier(out)
        if self.use_mid:
            if normals_s4 is not None:
                mid_in = torch.cat([out, normals_s4], dim=1)
            else:
                mid_in = F.pad(out, (0,0,0,0,0,3))
            result = result + self.mid_classifier(mid_in)
        return result

# --- YOLO Heads (unchanged structure, 35 classes) ---
class DecoupledHead(nn.Module):
    def __init__(self, ch_in, num_classes, width=128):
        super().__init__()
        self.cls_convs = nn.Sequential(DWSepConv(ch_in, width, 3, padding=1), nn.ReLU(inplace=True))
        self.reg_convs = nn.Sequential(DWSepConv(ch_in, width, 3, padding=1), nn.ReLU(inplace=True))
        self.cls_pred = nn.Conv2d(width, num_classes, 1)
        self.reg_pred = nn.Conv2d(width, 4, 1)
        self.obj_pred = nn.Conv2d(width, 1, 1)
    def forward(self, x):
        cls_feat = self.cls_convs(x); reg_feat = self.reg_convs(x)
        return torch.cat([self.reg_pred(reg_feat), self.obj_pred(reg_feat), self.cls_pred(cls_feat)], dim=1)

class YOLOHead(nn.Module):
    def __init__(self, fpn_ch=FPN_CH, num_classes=40):
        super().__init__()
        self.head_s8  = DecoupledHead(fpn_ch, num_classes, width=128)
        self.head_s16 = DecoupledHead(fpn_ch, num_classes, width=128)
        self.head_s32 = DecoupledHead(fpn_ch, num_classes, width=128)
    def forward(self, x_s8, x_s16, x_s32):
        return [self.head_s8(x_s8), self.head_s16(x_s16), self.head_s32(x_s32)]

# =====================================================================
# ✅ V2.7: FusedHexapodModel — 1-Channel Input, 5 Outputs
# =====================================================================
class FusedHexapodModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True,
                                           features_only=True, out_indices=(1, 2, 3, 4))
        feat_info = self.backbone.feature_info.channels()
        ch_s4, ch_s8, ch_s16, ch_s32 = feat_info

        # ✅ V2.7: Patch first conv to 1-channel input
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                              stride=old_conv.stride, padding=old_conv.padding, bias=False)
        # Merge RGB weights via luminance formula
        with torch.no_grad():
            w = old_conv.weight.data  # [C_out, 3, kH, kW]
            new_conv.weight.data = w[:, 0:1]*0.299 + w[:, 1:2]*0.587 + w[:, 2:3]*0.114
        self.backbone.conv_stem = new_conv
        print(f"  ✅ Backbone first conv: 3→1 channel (luminance merge)")

        disp_steps = config.get('internal_disp_steps', 48)
        self.stereo_head = HierarchicalStereoHead(ch_s8, ch_s4, max_disp_s8=disp_steps)
        self.normals_head = NormalsHead(ch_s4)
        self.seg_head = LRASPPHead(ch_s4, ch_s16, config['num_seg_classes'], normals_ch=3)
        self.fpn_neck = LightFPNNeck(ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH)
        self.yolo_head = YOLOHead(fpn_ch=FPN_CH, num_classes=config['num_det_classes'])

        total_params = sum(p.numel() for p in self.parameters())
        print(f"  Total parameters: {total_params:,}")
        print(f"  Normals Head: {sum(p.numel() for p in self.normals_head.parameters()):,}")
        print(f"  Seg Head: {sum(p.numel() for p in self.seg_head.parameters()):,}")
        print(f"  YOLO Head: {sum(p.numel() for p in self.yolo_head.parameters()):,}")

    def forward(self, x_left, x_right=None):
        # x_left: [B, 1, 480, 640] — single channel!
        features_l = self.backbone(x_left)
        # [0]=s4, [1]=s8, [2]=s16, [3]=s32

        # Stereo
        if x_right is not None:
            with torch.no_grad():
                features_r = self.backbone(x_right)
            final_disp, disp_s8 = self.stereo_head(
                features_l[1], features_r[1], features_l[0], x_left)
        else:
            final_disp, disp_s8 = None, None

        # Normals (from s4 features)
        normals = self.normals_head(features_l[0])

        # Segmentation (receives normals as extra input)
        seg = self.seg_head(features_l[0], features_l[2], normals_s4=normals)

        # YOLO
        fpn_s8, fpn_s16, fpn_s32 = self.fpn_neck(features_l[1], features_l[2], features_l[3])
        det = self.yolo_head(fpn_s8, fpn_s16, fpn_s32)

        return final_disp, seg, det, disp_s8, normals

model = FusedHexapodModel(CONFIG).to(DEVICE)
print(f"\n✅ V2.7 Model created on {DEVICE}")


## V2.7 Loss Functions
SimpleYOLOLoss (35 cls) + Stereo Smooth-L1 + Seg CrossEntropy + Normals Cosine

In [ ]:
# --- 1. Define Helper Blocks ---
class SimpleYOLOLoss(nn.Module):
    def __init__(self, num_classes, stride):
        super().__init__()
        self.num_classes = num_classes
        self.stride = stride
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
        self.l1 = nn.L1Loss(reduction='none')

    '''
    def _get_targets(self, targets, cls_preds, reg_preds):
        """
        Erzeugt die Target-Tensoren für Objectness, Regression und Klassifizierung.
        
        Encoding:
        - Offsets (dx, dy): Subpixel-Lage relativ zur Grid-Zelle
        - Größen (lw, lh): Logarithmierte Breite/Höhe in Grid-Einheiten
        """
        import math # Just in case it's not imported at the top of your file
        
        B, _, H, W = cls_preds.shape
        device = cls_preds.device
        
        # Initialisierung der Target-Tensoren
        cls_t = torch.zeros_like(cls_preds)
        reg_t = torch.zeros_like(reg_preds)
        obj_mask = torch.zeros((B, 1, H, W), device=device) 

        for b in range(B):
            if targets[b].numel() == 0: 
                continue
            
            # targets[b] Format: [N, 5] -> [class, xc, yc, w, h] (normalisiert 0-1)
            gt_boxes = targets[b].clone()
            
            # Koordinaten von Normalisiert [0, 1] auf Grid-Skala [0, W/H] umrechnen
            gt_boxes[:, 1] *= W  # xc in Grid-Einheiten
            gt_boxes[:, 2] *= H  # yc in Grid-Einheiten
            gt_boxes[:, 3] *= W  # w  in Grid-Einheiten
            gt_boxes[:, 4] *= H  # h  in Grid-Einheiten
            
            for box in gt_boxes:
                cls_id, gx, gy, gw, gh = box.tolist()
                
                # Grid-Zelle bestimmen (Integer-Anteil des wahren Zentrums)
                ix, iy = int(gx), int(gy)
                
                # Vorbereiten der logarithmischen Größen (für alle Nachbarn gleich)
                lw = math.log(max(gw, 1e-6))
                lh = math.log(max(gh, 1e-6))
                cid = int(cls_id)
                
                # 🚨 3x3 Cross Assignment (Zentrum + Oben, Unten, Links, Rechts)
                for dx, dy in [(0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nx, ny = ix + dx, iy + dy
                    
                    # Boundary Check: Liegt der Nachbar innerhalb des Grids?
                    if 0 <= nx < W and 0 <= ny < H:
                        
                        # 1. Objectness Target setzen
                        obj_mask[b, 0, ny, nx] = 1.0
                        
                        # 2. Regression Targets (dx, dy MÜSSEN relativ zum Nachbarn nx, ny sein!)
                        reg_t[b, 0, ny, nx] = gx - nx
                        reg_t[b, 1, ny, nx] = gy - ny
                        reg_t[b, 2, ny, nx] = lw
                        reg_t[b, 3, ny, nx] = lh
                        
                        # 3. Classification Target
                        if 0 <= cid < self.num_classes:
                            cls_t[b, cid, ny, nx] = 1.0

        return cls_t, reg_t, obj_mask
    '''
    def _get_targets(self, targets, cls_preds, reg_preds):
        """
        Vektorisierte Version:
        Erzeugt die Target-Tensoren für Objectness, Regression und Klassifizierung
        komplett ohne For-Schleifen über Boxen oder Nachbarn.
        """
        B, _, H, W = cls_preds.shape
        device = cls_preds.device
        
        # Initialisierung der Target-Tensoren
        cls_t = torch.zeros_like(cls_preds)
        reg_t = torch.zeros_like(reg_preds)
        obj_mask = torch.zeros((B, 1, H, W), device=device) 
        
        # 1. Targets flachklopfen und Batch-Index hinzufügen
        batch_targets = []
        for b in range(B):
            if targets[b].numel() > 0:
                t = targets[b].clone()
                # Batch-Index als erste Spalte hinzufügen: [b, class_id, xc, yc, w, h]
                b_idx = torch.full((t.shape[0], 1), b, device=device, dtype=t.dtype)
                batch_targets.append(torch.cat([b_idx, t], dim=1))
                
        # Wenn der ganze Batch leer ist, sind wir fertig
        if not batch_targets:
            return cls_t, reg_t, obj_mask
            
        gt = torch.cat(batch_targets, dim=0) # Shape: [Alle_Boxen_im_Batch, 6]
        
        # Variablen extrahieren (alles auf einmal!)
        b_idx  = gt[:, 0].long()
        cls_id = gt[:, 1].long()
        gx     = gt[:, 2] * W
        gy     = gt[:, 3] * H
        gw     = gt[:, 4] * W
        gh     = gt[:, 5] * H
        
        # Basis-Grid-Zelle
        ix = gx.long()
        iy = gy.long()
        
        # 2. 5 Nachbarn generieren (Cross Assignment)
        # Offsets für Zentrum, Links, Rechts, Oben, Unten
        offsets = torch.tensor([
            [0, 0], [-1, 0], [1, 0], [0, -1], [0, 1]
        ], device=device, dtype=torch.long)
        
        # Wir berechnen die Nachbar-Koordinaten (nx, ny) für alle Boxen gleichzeitig
        # unsqueeze() hilft uns, die Matrix zu erweitern und offsets zu addieren
        nx = (ix.unsqueeze(1) + offsets[:, 0].unsqueeze(0)).flatten()
        ny = (iy.unsqueeze(1) + offsets[:, 1].unsqueeze(0)).flatten()
        
        # Da wir nun aus jeder Box 5 Nachbarn gemacht haben, 
        # müssen wir die anderen Werte (Batch-ID, Zielwerte) ebenfalls 5-mal wiederholen.
        b_idx_rep  = b_idx.repeat_interleave(5)
        cls_id_rep = cls_id.repeat_interleave(5)
        gx_rep     = gx.repeat_interleave(5)
        gy_rep     = gy.repeat_interleave(5)
        gw_rep     = gw.repeat_interleave(5)
        gh_rep     = gh.repeat_interleave(5)
        
        # 3. Boundary Check AND Class Check
        num_classes = cls_preds.shape[1]  # Hole die Anzahl der Klassen vom Tensor
        
        valid_mask = (
            (nx >= 0) & (nx < W) & 
            (ny >= 0) & (ny < H) & 
            (cls_id_rep >= 0) & (cls_id_rep < num_classes) # 🚨 NEU: Verhindert CUDA Crash!
        )
        
        # Wir behalten nur die Werte, wo valid_mask True ist
        b_v   = b_idx_rep[valid_mask]
        c_v   = cls_id_rep[valid_mask]
        nx_v  = nx[valid_mask]
        ny_v  = ny[valid_mask]
        gx_v  = gx_rep[valid_mask]
        gy_v  = gy_rep[valid_mask]
        gw_v  = gw_rep[valid_mask]
        gh_v  = gh_rep[valid_mask]
        
        # 4. Vorbereiten der logarithmischen Größen
        lw_v = torch.log(torch.clamp(gw_v, min=1e-6))
        lh_v = torch.log(torch.clamp(gh_v, min=1e-6))
        
        # 5. Werte in die Tensoren schreiben
        obj_mask[b_v, 0, ny_v, nx_v] = 1.0
        
        reg_t[b_v, 0, ny_v, nx_v] = gx_v - nx_v
        reg_t[b_v, 1, ny_v, nx_v] = gy_v - ny_v
        reg_t[b_v, 2, ny_v, nx_v] = lw_v
        reg_t[b_v, 3, ny_v, nx_v] = lh_v
        
        cls_t[b_v, c_v, ny_v, nx_v] = 1.0
        
        return cls_t, reg_t, obj_mask
        
    def sigmoid_focal_loss(self, inputs, targets, alpha=0.75, gamma=2.0, reduction='none'): # ⬅️ Changed default to 0.75
        p = torch.sigmoid(inputs)
        ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
        p_t = p * targets + (1 - p) * (1 - targets)
        loss = ce_loss * ((1 - p_t) ** gamma)

        if alpha >= 0:
            # If target=1, alpha_t = 0.75. If target=0, alpha_t = 0.25. 
            # This properly boosts the rare positive anchors!
            alpha_t = alpha * targets + (1 - alpha) * (1 - targets) 
            loss = alpha_t * loss

        if reduction == "mean":
            return loss.mean()
        elif reduction == "sum":
            return loss.sum()
        else:
            return loss

    def forward(self, preds, targets):
        # Slice components: [B, 5 + Num_Classes, H, W]
        reg_p = preds[:, :4, :, :]   # [dx, dy, log_w, log_h]
        obj_p = preds[:, 4:5, :, :]  # [objectness]
        cls_p = preds[:, 5:, :, :]   # [classes]

        # 1. Get Ground Truth Targets
        cls_t, reg_t, obj_mask = self._get_targets(targets, cls_p, reg_p)
        
        # 🚨 METRICS EXTRACTION FOR TENSORBOARD
        num_targets_val = obj_mask.sum().item()
        max_cls_val = cls_t.max().item() if num_targets_val > 0 else 0

        num_pos = torch.clamp(obj_mask.sum(), min=1.0)

        # 🚨 Protect against empty TartanAir targets overwriting the weights
        if num_targets_val == 0:
            return {
                'total': torch.tensor(0.0, requires_grad=True, device=preds.device),
                'box': 0.0, 'obj': 0.0, 'cls': 0.0,
                'num_targets': 0,     # Added
                'max_cls': 0          # Added
            }
        
        # 2. OBJECTNESS LOSS (mit Focal Loss für bessere Balance)
        l_obj = self.sigmoid_focal_loss(obj_p, obj_mask, reduction='sum') / num_pos

        # 3. REGRESSION LOSS (nur auf positiven Samples)
        # WICHTIG: Separate Gewichtung für Offsets vs. Größen
        # VORAUSSETZUNG: self.l1 = nn.L1Loss(reduction='none')
        l_offset = (self.l1(reg_p[:, :2], reg_t[:, :2]) * obj_mask).sum() / num_pos
        l_size = (self.l1(reg_p[:, 2:], reg_t[:, 2:]) * obj_mask).sum() / num_pos
        l_box = l_offset + 2.0 * l_size  # Größen sind wichtiger

        # 4. CLASSIFICATION LOSS (BUG FIXED: Nur auf positiven Samples!)
        l_cls_raw = self.sigmoid_focal_loss(cls_p, cls_t, reduction='none')
        l_cls = (l_cls_raw * obj_mask).sum() / num_pos

        # ANGEPASSTE Gewichtung: Box-Loss ist jetzt wichtiger
        return {
            'total': 5.0 * l_box + 5.0 * l_obj + l_cls, 
            'box': l_box.item(),
            'obj': l_obj.item(),
            'cls': l_cls.item(),
            'num_targets': num_targets_val,  # ⬅️ To TensorBoard
            'max_cls': max_cls_val           # ⬅️ To TensorBoard
        }

class EdgeAwareSmoothnessLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, disp, img):
        mean_disp = disp.mean(2, True).mean(3, True)
        disp = disp / (mean_disp + 1e-7)
        
        grad_disp_x = torch.abs(disp[:, :, :, :-1] - disp[:, :, :, 1:])
        grad_disp_y = torch.abs(disp[:, :, :-1, :] - disp[:, :, 1:, :])

        grad_img_x = torch.mean(torch.abs(img[:, :, :, :-1] - img[:, :, :, 1:]), 1, keepdim=True)
        grad_img_y = torch.mean(torch.abs(img[:, :, :-1, :] - img[:, :, 1:, :]), 1, keepdim=True)

        grad_img_x = torch.exp(-torch.mean(grad_img_x, 1, keepdim=True))
        grad_img_y = torch.exp(-torch.mean(grad_img_y, 1, keepdim=True))

        return torch.mean(grad_disp_x * grad_img_x) + torch.mean(grad_disp_y * grad_img_y)


# --- 2. Main Loss Class ---
class FusedHexapodLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        # YOLO Skalen-Verluste
        self.yolo_s8_loss  = SimpleYOLOLoss(config['num_det_classes'], stride=8)
        self.yolo_s16_loss = SimpleYOLOLoss(config['num_det_classes'], stride=16)
        self.yolo_s32_loss = SimpleYOLOLoss(config['num_det_classes'], stride=32)
        
        # KEIN seg_loss (CrossEntropy gegen GT) mehr hier!
        self.kl_div = nn.KLDivLoss(reduction='batchmean') 
        self.smooth_loss = EdgeAwareSmoothnessLoss()
        
        # Basis-Gewichte (statisch in StaticWeightedLoss)
        self.w_yolo = 1.0 
        self.w_kd = 1.0     # Fokus liegt jetzt auf KD für Segmentation
        self.w_stereo = 1.0

    def forward(self, preds, targets, teacher_preds, left_img=None):
        # Entpacken der Vorhersagen
        stereo_preds, seg_preds, det_preds = preds
        
        task_tensors = {} # Für Gradienten (Tensors)
        logs = {}         # Für Tensorboard (Floats)

        # --- 1. STEREO LOSS ---
        if stereo_preds is not None and targets.get('disp') is not None:
            gt_disp = targets['disp']
            if gt_disp.dim() == 3: gt_disp = gt_disp.unsqueeze(1)
            
            mask = (gt_disp > 0) & (gt_disp < CONFIG['max_disp_pixel'])
            if mask.sum() > 0:
                
                # 1. Erst plain L1 für die Beta-Schätzung
                with torch.no_grad():
                    current_beta = max(1.0, F.l1_loss(stereo_preds[mask], gt_disp[mask]).item() * 0.5)
                # 2. Dann Smooth L1 mit dynamischem Beta
                l_stereo_raw = F.smooth_l1_loss(stereo_preds[mask], gt_disp[mask], beta=current_beta)
                # 2. Add Edge-Aware Smoothness
                if left_img is not None:
                    l_smooth = self.smooth_loss(stereo_preds, left_img)
                    
                    # FIX: Instead of a hard 0.02, scale it relative to the raw L1 loss.
                    # This prevents the smoothness penalty from dominating when the network
                    # gets close to convergence, allowing sharp details to emerge.
                    dynamic_smooth_weight = torch.clamp(l_stereo_raw.detach() * 0.01, max=0.02)
                    l_stereo = l_stereo_raw + dynamic_smooth_weight * l_smooth
                    
                    logs['smooth'] = l_smooth.item()
                else:
                    l_stereo = l_stereo_raw
                
                # ✅ FIX: /2→/3 — etwas stärker gedämpft, damit Stereo nicht YOLO dominiert
                task_tensors['stereo'] = l_stereo / 3.0
                logs['stereo'] = l_stereo.item()

        # --- 2. KNOWLEDGE DISTILLATION (KD) ---
        if teacher_preds is not None and seg_preds is not None:
            T = 2.0 
            
            # Student Logits auf Teacher-Größe
            s_logits = F.interpolate(seg_preds, size=teacher_preds.shape[-2:], 
                                     mode='bilinear', align_corners=False)
            
            # STUDENT: Log-Softmax (Korrekt für KLDiv)
            p_s = F.log_softmax(s_logits / T, dim=1)
            
            # TEACHER: Numerisch stabilere Berechnung
            # Wir nehmen an, teacher_preds sind LOGITS (rohe Scores).
            # Statt exp() -> log() -> softmax() machen wir direkt Softmax auf den Logits.
            p_t = F.softmax(teacher_preds / T, dim=1)
            
            # KL Divergenz BERECHNUNG
            # WICHTIG: reduction='none', damit wir selbst über Pixel mitteln können
            kl_loss_pixelwise = F.kl_div(p_s, p_t, reduction='none') * (T**2)
            
            # Jetzt die Magie: 
            # 1. Summe über Klassen (dim=1) -> Das ist die KL-Div pro Pixel
            # 2. Mittelwert über Spatial (dim=2,3) & Batch (dim=0) -> Skalenunabhängig!
            l_kd = kl_loss_pixelwise.sum(dim=1).mean()
            
            task_tensors['seg'] = l_kd 
            logs['seg_kd'] = l_kd.item()
            
        # --- 3. YOLO LOSS ---
        if det_preds is not None and targets.get('det') is not None:
            gt_det = targets['det']
            
            # Alle 3 Skalen berechnen
            r8  = self.yolo_s8_loss(det_preds[0], gt_det)
            r16 = self.yolo_s16_loss(det_preds[1], gt_det)
            r32 = self.yolo_s32_loss(det_preds[2], gt_det)
            
            l_yolo = (r8['total'] + r16['total'] + r32['total']) / 3.0
            
            # ✅ FIX: /10→/4 — Phase 1 hatte YOLO um Faktor 2.5x zu stark gedämpft
            # Resultat: YOLO Gradient Norm war 10-18x kleiner als Stereo
            task_tensors['yolo'] = l_yolo / 4.0
            logs['yolo'] = l_yolo.item()
            
            # Helper to safely extract floats whether it's a tensor or a primitive float
            def _to_float(val): return val.item() if isinstance(val, torch.Tensor) else float(val)
            
            logs['yolo_box'] = _to_float((r8['box'] + r16['box'] + r32['box']) / 3.0)
            logs['yolo_obj'] = _to_float((r8['obj'] + r16['obj'] + r32['obj']) / 3.0)
            logs['yolo_cls'] = _to_float((r8['cls'] + r16['cls'] + r32['cls']) / 3.0)

            # 🚨 FIX: Extract the debug metrics and pass them up!
            # Sum the targets across all 3 scales
            logs['yolo_num_targets'] = r8.get('num_targets', 0) + r16.get('num_targets', 0) + r32.get('num_targets', 0)
            
            # Get the highest class ID found across all 3 scales
            logs['yolo_max_cls'] = max(r8.get('max_cls', 0), r16.get('max_cls', 0), r32.get('max_cls', 0))

        return task_tensors, logs
print("✅ Loss function updated.")

In [ ]:
# =====================================================================
# V2.7 Losses: EMA-Balanced Multi-Task
# =====================================================================

class NormalsLoss(nn.Module):
    """Angular loss for surface normals — stronger gradient than cosine near convergence."""
    def forward(self, pred, gt, valid_mask):
        cosine_sim = (pred * gt).sum(dim=1, keepdim=True)
        cosine_sim = torch.clamp(cosine_sim, -1.0 + 1e-6, 1.0 - 1e-6)
        angular_error = torch.acos(cosine_sim)
        loss = angular_error * valid_mask
        return loss.sum() / (valid_mask.sum() + 1e-6)

class EdgeAwareSmoothnessLoss(nn.Module):
    def forward(self, disp, img):
        mean_disp = disp.mean(2, True).mean(3, True)
        disp = disp / (mean_disp + 1e-7)
        grad_disp_x = torch.abs(disp[:,:,:,:-1] - disp[:,:,:,1:])
        grad_disp_y = torch.abs(disp[:,:,:-1,:] - disp[:,:,1:,:])
        grad_img_x = torch.mean(torch.abs(img[:,:,:,:-1] - img[:,:,:,1:]), 1, keepdim=True)
        grad_img_y = torch.mean(torch.abs(img[:,:,:-1,:] - img[:,:,1:,:]), 1, keepdim=True)
        return (grad_disp_x * torch.exp(-grad_img_x)).mean() + \
               (grad_disp_y * torch.exp(-grad_img_y)).mean()


class V27HexapodLoss(nn.Module):
    """
    V2.7 Loss with EMA-based gradient balancing.
    
    Each task loss is normalized by its exponential moving average (EMA).
    This ensures all tasks contribute ~equally regardless of absolute loss scale.
    
    Priority weights control relative importance:
      priority=1.0 → equal contribution
      priority=1.5 → 50% more gradient budget than others
    
    No trainable parameters. No separate optimizer. EMA adapts immediately.
    """
    def __init__(self, config):
        super().__init__()
        self.yolo_s8  = SimpleYOLOLoss(config['num_det_classes'], stride=8)
        self.yolo_s16 = SimpleYOLOLoss(config['num_det_classes'], stride=16)
        self.yolo_s32 = SimpleYOLOLoss(config['num_det_classes'], stride=32)
        self.smooth_loss = EdgeAwareSmoothnessLoss()
        self.normals_loss = NormalsLoss()
        weights = torch.tensor(config['seg_class_weights'], dtype=torch.float32)
        self.seg_ce = nn.CrossEntropyLoss(weight=weights, ignore_index=255)

        # EMA buffers (saved in checkpoint, no gradient)
        self.ema_decay = 0.99  # Adapts over ~100 steps
        self.register_buffer('ema_stereo',  torch.tensor(1.0))
        self.register_buffer('ema_yolo',    torch.tensor(1.0))
        self.register_buffer('ema_seg',     torch.tensor(1.0))
        self.register_buffer('ema_normals', torch.tensor(1.0))
        self.ema_initialized = False

        # Priority: how IMPORTANT is each task (not scale!)
        self.priority = {'stereo': 1.0, 'yolo': 1.5, 'seg': 1.0, 'normals': 1.0}

    def _update_ema(self, name, value):
        ema = getattr(self, f'ema_{name}')
        if not self.ema_initialized:
            ema.fill_(value)
        else:
            ema.mul_(self.ema_decay).add_(value * (1.0 - self.ema_decay))

    def forward(self, preds, targets, left_img=None, coarse_disp=None):
        disp_pred, seg_pred, det_pred, _, normals_pred = preds
        task_losses, logs = {}, {}

        # --- STEREO ---
        if disp_pred is not None and targets.get('disp') is not None:
            gt_disp = targets['disp']
            if gt_disp.dim() == 3: gt_disp = gt_disp.unsqueeze(1)
            mask = (gt_disp > 0) & (gt_disp < CONFIG['max_disp_pixel'])
            if mask.sum() > 0:
                with torch.no_grad():
                    beta = max(1.0, F.l1_loss(disp_pred[mask], gt_disp[mask]).item() * 0.5)
                l_stereo = F.smooth_l1_loss(disp_pred[mask], gt_disp[mask], beta=beta)
                if left_img is not None:
                    l_sm = self.smooth_loss(disp_pred, left_img)
                    l_stereo = l_stereo + torch.clamp(l_stereo.detach()*0.01, max=0.02) * l_sm
                    logs['smooth'] = l_sm.item()
                if coarse_disp is not None:
                    gt_s8 = F.interpolate(gt_disp, size=coarse_disp.shape[-2:], mode='nearest') / 8.0
                    m8 = (gt_s8 > 0) & (gt_s8 < 24)
                    if m8.sum() > 0:
                        l_aux = F.smooth_l1_loss(coarse_disp[m8], gt_s8[m8], beta=0.5)
                        l_stereo = l_stereo + 0.1 * l_aux
                        logs['stereo_aux'] = l_aux.item()
                task_losses['stereo'] = l_stereo
                logs['stereo'] = l_stereo.item()

        # --- NORMALS ---
        if normals_pred is not None and targets.get('normals') is not None:
            gt_normals = targets['normals']
            valid_mask = targets.get('normals_valid', (gt_normals.abs().sum(dim=1, keepdim=True) > 0.5).float())
            if normals_pred.shape[-2:] != gt_normals.shape[-2:]:
                gt_normals = F.interpolate(gt_normals, size=normals_pred.shape[-2:], mode='bilinear', align_corners=False)
                gt_normals = F.normalize(gt_normals, dim=1, eps=1e-6)
                valid_mask = F.interpolate(valid_mask, size=normals_pred.shape[-2:], mode='nearest')
            l_normals = self.normals_loss(normals_pred, gt_normals, valid_mask)
            task_losses['normals'] = l_normals
            logs['normals'] = l_normals.item()

        # --- SEG ---
        if seg_pred is not None and targets.get('seg') is not None:
            gt_seg = targets['seg']
            if seg_pred.shape[-2:] != gt_seg.shape[-2:]:
                seg_pred_up = F.interpolate(seg_pred, size=gt_seg.shape[-2:], mode='bilinear', align_corners=False)
            else:
                seg_pred_up = seg_pred
            l_seg = self.seg_ce(seg_pred_up, gt_seg)
            task_losses['seg'] = l_seg
            logs['seg_ce'] = l_seg.item()

        # --- YOLO ---
        if det_pred is not None and targets.get('det') is not None:
            gt_det = targets['det']
            r8  = self.yolo_s8(det_pred[0], gt_det)
            r16 = self.yolo_s16(det_pred[1], gt_det)
            r32 = self.yolo_s32(det_pred[2], gt_det)
            l_yolo = (r8['total'] + r16['total'] + r32['total']) / 3.0
            task_losses['yolo'] = l_yolo
            logs['yolo'] = l_yolo.item()
            _f = lambda v: v.item() if isinstance(v, torch.Tensor) else float(v)
            logs['yolo_box'] = _f((r8['box']+r16['box']+r32['box'])/3)
            logs['yolo_obj'] = _f((r8['obj']+r16['obj']+r32['obj'])/3)
            logs['yolo_cls'] = _f((r8['cls']+r16['cls']+r32['cls'])/3)
            logs['yolo_num_targets'] = r8.get('num_targets',0)+r16.get('num_targets',0)+r32.get('num_targets',0)

        # ✅ EMA-NORMALIZED WEIGHTING
        # Update EMAs (no gradient)
        with torch.no_grad():
            for name in task_losses:
                self._update_ema(name, task_losses[name].detach())
            self.ema_initialized = True

        # Normalize + weight
        total = sum(
            self.priority[name] * loss / getattr(self, f'ema_{name}').clamp(min=1e-6)
            for name, loss in task_losses.items()
        )

        # Log effective weights and EMAs
        for name in task_losses:
            ema_val = getattr(self, f'ema_{name}').item()
            logs[f'ema/{name}'] = ema_val
            logs[f'weight_effective/{name}'] = self.priority[name] / max(ema_val, 1e-6)

        return total, logs

print("\u2705 V2.7 Loss with EMA balancing loaded")
print("  Priority: stereo=1.0, yolo=1.5, seg=1.0, normals=1.0")
print("  EMA decay=0.99 (~100 step adaptation)")


## V2.7 Data: Parsers + On-the-fly Normals/Seg

In [ ]:
import os
import glob
import numpy as np
from pycocotools.coco import COCO

def parse_tartan(root):
    """Scan TartanAir for stereo pairs + depth (normals+seg computed on-the-fly)."""
    print("   Scanning TartanAir...")
    samples = []
    for l_folder in glob.glob(os.path.join(root, '**', 'image_left'), recursive=True):
        parent = os.path.dirname(l_folder)
        for f in sorted(os.listdir(l_folder)):
            if not f.endswith('.png') or 'Zone.Identifier' in f: continue
            l_path = os.path.join(l_folder, f)
            r_path = os.path.join(parent, 'image_right', f.replace('_left', '_right'))
            d_path = os.path.join(parent, 'depth_left', f.replace('.png', '_depth.npy'))
            if os.path.exists(r_path) and os.path.exists(d_path):
                samples.append({'type':'stereo', 'source':'tartan',
                                'l': l_path, 'r': r_path, 'd': d_path, 'b': None})
    print(f"   -> Found {len(samples)} TartanAir samples.")
    return samples

def parse_coco_robot35(root, ann_file=None):
    """
    Load COCO with 35 robot-relevant classes.
    Uses pseudo-labels JSON if available, otherwise standard COCO GT.
    """
    print("   Scanning COCO (35 robot classes)...")
    if ann_file is None:
        # Try pseudo-labels first, fallback to standard GT
        pseudo_path = os.path.join(root, 'annotations', 'instances_train2017_robot35_pseudo.json')
        gt_path = os.path.join(root, 'annotations', 'instances_train2017.json')
        if os.path.exists(pseudo_path):
            ann_file = pseudo_path
            print(f"   Using pseudo-labels: {pseudo_path}")
        else:
            ann_file = gt_path
            print(f"   ⚠️ No pseudo-labels found, using raw GT: {gt_path}")

    coco = COCO(ann_file)
    img_ids = coco.getImgIds()
    samples = []

    for img_id in img_ids:
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(root, 'train2017', img_info['file_name'])
        if not os.path.exists(img_path): continue

        anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
        boxes = []
        for ann in anns:
            cat_id = ann['category_id']
            if cat_id not in robot_cat_to_continuous: continue
            cls_idx = robot_cat_to_continuous[cat_id]
            x, y, w, h = ann['bbox']
            img_w, img_h = img_info['width'], img_info['height']
            cx = (x + w/2) / img_w
            cy = (y + h/2) / img_h
            nw = w / img_w
            nh = h / img_h
            if nw > 0.001 and nh > 0.001:
                boxes.append([cls_idx, cx, cy, nw, nh])

        samples.append({'type':'det', 'source':'coco', 'l': img_path,
                        'b': boxes if boxes else None})

    n_with_boxes = sum(1 for s in samples if s['b'] is not None)
    print(f"   -> Found {len(samples)} COCO images ({n_with_boxes} with robot-class boxes)")
    return samples

print("✅ V2.7 Parsers loaded")


In [ ]:
def read_pfm_fixed(file_path):
    """Reads a .pfm file and returns a numpy array."""
    with open(file_path, 'rb') as f:
        header = f.readline().decode().rstrip()
        color = (header == 'PF')
        dim_match = re.match(r'^(\d+)\s(\d+)\s$', f.readline().decode('utf-8'))
        width, height = map(int, dim_match.groups())
        scale = float(f.readline().decode().rstrip())
        endian = '<' if scale < 0 else '>'
        scale = abs(scale)
        data = np.fromfile(f, endian + 'f')
        shape = (height, width, 3) if color else (height, width)
        data = np.reshape(data, shape)
        data = np.flipud(data)
    return (data * scale).copy()

# =====================================================================
# On-the-fly Normals + Seg computation
# =====================================================================
def depth_to_normals_np(depth, fx, fy):
    """Compute surface normals from depth map. Returns (normals[3,H,W], valid[H,W])."""
    dz_dx = np.zeros_like(depth)
    dz_dy = np.zeros_like(depth)
    dz_dx[:, 1:-1] = (depth[:, 2:] - depth[:, :-2]) / 2.0
    dz_dy[1:-1, :] = (depth[2:, :] - depth[:-2, :]) / 2.0
    dz_dx[:, 0] = depth[:, 1] - depth[:, 0]
    dz_dx[:, -1] = depth[:, -1] - depth[:, -2]
    dz_dy[0, :] = depth[1, :] - depth[0, :]
    dz_dy[-1, :] = depth[-1, :] - depth[-2, :]

    nx = -dz_dx * fy
    ny = -dz_dy * fx
    nz = np.ones_like(nx) * (fx * fy / 1000.0)
    norm = np.sqrt(nx**2 + ny**2 + nz**2 + 1e-8)
    normals = np.stack([nx/norm, ny/norm, nz/norm], axis=0).astype(np.float32)
    valid = (depth > 0.01) & (depth < 100.0) & (np.abs(dz_dx) < 5.0) & (np.abs(dz_dy) < 5.0)
    normals[:, ~valid] = 0.0
    return normals, valid

def compute_seg6_from_depth_normals(depth, normals):
    """Compute 6-class seg from depth + normals. Returns [H,W] uint8."""
    ny = normals[1]
    cos_angle = np.clip(np.abs(ny), 0, 1)
    angle = np.arccos(cos_angle) * 180.0 / np.pi  # 0=horizontal, 90=vertical

    grad_x = scipy_sobel(depth, axis=1)
    grad_y = scipy_sobel(depth, axis=0)
    depth_grad = np.sqrt(grad_x**2 + grad_y**2)

    valid = (depth > 0.01) & (depth < 100.0)
    nvalid = np.sqrt(normals[0]**2 + normals[1]**2 + normals[2]**2) > 0.5

    seg = np.full(depth.shape, 5, dtype=np.uint8)  # VOID
    walkable = valid & nvalid & (angle < 25) & (depth > 0.1) & (depth < 8.0)
    seg[walkable] = 0
    seg[walkable & (depth_grad > 0.05)] = 1  # STEP
    wall = valid & nvalid & (angle > 60) & (depth > 0.1) & (depth < 15.0)
    seg[wall] = 2
    obstacle = valid & nvalid & (~walkable) & (~wall) & (angle >= 25) & (angle <= 60) & (depth > 0.1) & (depth < 4.0)
    seg[obstacle] = 3

    # Vegetation approximation
    depth_mean = uniform_filter(depth, size=3)
    depth_sq_mean = uniform_filter(depth**2, size=3)
    local_var = np.clip(depth_sq_mean - depth_mean**2, 0, None)
    veg = valid & nvalid & (angle > 30) & (angle < 65) & (depth > 0.5) & (depth < 10.0) & (local_var > 0.01)
    seg[veg] = 4
    seg[~valid] = 5
    return seg


class V27Dataset(Dataset):
    """V2.7 Dataset: 1-channel input, on-the-fly normals+seg from depth."""
    def __init__(self, roots, img_size=(480, 640)):
        self.img_size = img_size
        self.samples = []
        if 'tartan' in roots:
            self.samples.extend(parse_tartan(roots['tartan']))
        if 'coco' in roots:
            self.samples.extend(parse_coco_robot35(roots['coco']))

        # ✅ V2.7: Single-channel normalization
        self.gray_mean = 0.449  # 0.485*0.299 + 0.456*0.587 + 0.406*0.114
        self.gray_std  = 0.226  # Approximation for luminance

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        source = sample.get('source')
        cv2_size = (self.img_size[1], self.img_size[0])

        if source == 'tartan':
            img_l = cv2.imread(sample['l'])
            img_r = cv2.imread(sample['r'])
            if img_l is None or img_r is None:
                return self.__getitem__(0)

            img_l = cv2.resize(img_l, cv2_size)
            img_r = cv2.resize(img_r, cv2_size)

            # ✅ V2.7: TRUE single-channel input
            gray_l = cv2.cvtColor(img_l, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            gray_r = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            t_l = torch.from_numpy((gray_l - self.gray_mean) / self.gray_std).unsqueeze(0)  # [1, H, W]
            t_r = torch.from_numpy((gray_r - self.gray_mean) / self.gray_std).unsqueeze(0)

            # Depth → Disparity
            raw_depth = np.load(sample['d']).astype(np.float32)
            if raw_depth.shape != (self.img_size[0], self.img_size[1]):
                raw_depth = cv2.resize(raw_depth, cv2_size, interpolation=cv2.INTER_NEAREST)
            valid_d = raw_depth > 1e-4
            disp = np.zeros_like(raw_depth)
            disp[valid_d] = 80.0 / raw_depth[valid_d]
            disp = np.clip(disp, 0, CONFIG['max_disp_pixel'])
            disp_tensor = torch.from_numpy(disp).unsqueeze(0).float()

            # ✅ V2.7: On-the-fly normals from depth
            normals, normals_valid = depth_to_normals_np(raw_depth, CONFIG['tartan_fx'], CONFIG['tartan_fy'])
            normals_tensor = torch.from_numpy(normals).float()  # [3, H, W]
            normals_valid_tensor = torch.from_numpy(normals_valid.astype(np.float32)).unsqueeze(0)  # [1, H, W]

            # ✅ V2.7: On-the-fly seg labels from depth+normals
            seg = compute_seg6_from_depth_normals(raw_depth, normals)
            seg_tensor = torch.from_numpy(seg).long()  # [H, W]

            # Teacher input for SegFormer (if still needed for comparison)
            img_l_rgb = cv2.cvtColor(img_l, cv2.COLOR_BGR2RGB)
            t_rgb = torch.from_numpy(img_l_rgb).permute(2,0,1).float() / 255.0
            imagenet_mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
            imagenet_std = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
            teacher_input = (t_rgb - imagenet_mean) / imagenet_std

            return {
                'left': t_l, 'right': t_r, 'teacher': teacher_input,
                'disp': disp_tensor, 'seg': seg_tensor,
                'normals': normals_tensor, 'normals_valid': normals_valid_tensor,
                'det': None, 'use_stereo': 1.0, 'use_seg': 1.0, 'use_yolo': 0.0
            }

        elif source == 'coco':
            img_raw = cv2.imread(sample['l'])
            if img_raw is None:
                return self.__getitem__(0)
            img_resized = cv2.resize(img_raw, cv2_size)

            # ✅ V2.7: Single-channel
            gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
            t_l = torch.from_numpy((gray - self.gray_mean) / self.gray_std).unsqueeze(0)

            # Teacher (RGB for optional comparison)
            img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
            t_rgb = torch.from_numpy(img_rgb).permute(2,0,1).float() / 255.0
            imagenet_mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
            imagenet_std = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
            teacher_input = (t_rgb - imagenet_mean) / imagenet_std

            det_tensor = torch.zeros((0, 5))
            if sample.get('b'):
                raw_boxes = np.array(sample['b'])
                if len(raw_boxes) > 0:
                    valid = (raw_boxes[:, 3] > 1e-4) & (raw_boxes[:, 4] > 1e-4)
                    clean = raw_boxes[valid]
                    if len(clean) > 0:
                        clean[:, 1:] = np.clip(clean[:, 1:], 0.0, 1.0)
                        det_tensor = torch.tensor(clean, dtype=torch.float32)

            return {
                'left': t_l, 'right': t_l.clone(), 'teacher': teacher_input,
                'disp': None, 'seg': None,
                'normals': None, 'normals_valid': None,
                'det': det_tensor, 'use_stereo': 0.0, 'use_seg': 0.0, 'use_yolo': 1.0
            }

print("✅ V27Dataset loaded (1-channel, on-the-fly normals+seg)")


## V2.7 Visualizer + Dashboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import PowerNorm
import torchvision

SEG_COLORS = np.array([
    [0, 200, 0],    [255, 165, 0],  [100, 100, 200],
    [200, 50, 50],  [0, 150, 0],    [50, 50, 50]
], dtype=np.uint8)

def visualize_v27(step, writer):
    """V2.7 Dashboard: 2 rows × 5 cols."""
    was_training = model.training
    model.eval()
    VIS_THRESH = 0.30

    with torch.no_grad():
        fig, axes = plt.subplots(2, 5, figsize=(30, 12))

        for row, (name, b) in enumerate(static_batches.items()):
            l = b['left'].to(DEVICE)
            r = b['right'].to(DEVICE)

            final_disp, seg_preds, det_preds, _, normals_pred = model(l, r)

            # Denormalize for display
            img_display = l[0, 0].cpu().numpy()  # Single channel
            img_display = (img_display * 0.226 + 0.449)  # Undo normalization
            img_display = np.clip(img_display, 0, 1)

            disp_gt = b.get('disp')

            # Col 0: Input Image (grayscale)
            ax = axes[row, 0]
            ax.imshow(img_display, cmap='gray')
            if name == 'yolo' and 'det' in b and b['det'] is not None and len(b['det']) > 0 and b['det'][0] is not None:
                h, w = img_display.shape
                for box in b['det'][0]:
                    cls_id, xc, yc, bw, bh = box.tolist()
                    color = plt.cm.tab20(int(cls_id) % 20)
                    rect = plt.Rectangle(((xc-bw/2)*w, (yc-bh/2)*h), bw*w, bh*h,
                                          fill=False, edgecolor=color, linewidth=1.5)
                    ax.add_patch(rect)
            ax.set_title("Input" if name != 'yolo' else "Input + GT Boxes", fontsize=10)
            ax.axis('off')

            # Col 1: Stereo
            ax = axes[row, 1]
            if name != 'yolo':
                disp_gt_np = disp_gt[0].squeeze().cpu().numpy() if disp_gt is not None else np.zeros_like(img_display)
                vmax = np.percentile(disp_gt_np[disp_gt_np > 0], 98) if (disp_gt_np > 0).any() else 50
                disp_np = final_disp[0].squeeze().cpu().numpy() if final_disp is not None else np.zeros_like(img_display)
                ax.imshow(disp_np, cmap='magma', vmin=0, vmax=vmax)
                ax.set_title("Predicted Disparity", fontsize=10)
            else:
                # YOLO confidence map
                yolo_map = det_preds[0][0]
                obj = torch.sigmoid(yolo_map[4])
                cls_max = torch.sigmoid(yolo_map[5:]).max(dim=0)[0]
                conf = (obj * cls_max).cpu().numpy()
                ax.imshow(conf, cmap='turbo', norm=PowerNorm(gamma=0.3, vmin=0, vmax=1))
                ax.set_title(f"YOLO Conf (max={conf.max():.2f})", fontsize=10)
            ax.axis('off')

            # Col 2: Segmentation (6 classes)
            ax = axes[row, 2]
            seg_np = torch.argmax(seg_preds.float(), dim=1)[0].cpu().numpy()
            seg_rgb = SEG_COLORS[seg_np]
            ax.imshow(seg_rgb)
            ax.set_title("Seg (6 classes)", fontsize=10)
            ax.axis('off')

            # Col 3: Surface Normals
            ax = axes[row, 3]
            n_np = normals_pred[0].cpu().numpy()
            normals_viz = ((n_np.transpose(1,2,0) + 1.0) * 127.5).clip(0,255).astype(np.uint8)
            # Upsample to full res for display
            normals_viz = cv2.resize(normals_viz, (img_display.shape[1], img_display.shape[0]))
            ax.imshow(normals_viz)
            ax.set_title("Surface Normals (RGB)", fontsize=10)
            ax.axis('off')

            # Col 4: YOLO boxes (with NMS) or GT Disparity
            ax = axes[row, 4]
            if name == 'yolo':
                ax.imshow(img_display, cmap='gray')
                h, w = img_display.shape
                yolo_map = det_preds[0][0]; stride = 8
                obj = torch.sigmoid(yolo_map[4])
                cls_p = torch.sigmoid(yolo_map[5:])
                final_conf = obj * cls_p.max(dim=0)[0]
                mask = final_conf > VIS_THRESH
                if mask.sum() > 0:
                    ys, xs = torch.where(mask)
                    all_boxes, all_scores, all_cls = [], [], []
                    for i in range(len(xs)):
                        gx, gy = xs[i].item(), ys[i].item()
                        dx = torch.sigmoid(yolo_map[0,gy,gx]).item()
                        dy = torch.sigmoid(yolo_map[1,gy,gx]).item()
                        gw = torch.exp(torch.clamp(yolo_map[2,gy,gx],-5,5)).item()
                        gh = torch.exp(torch.clamp(yolo_map[3,gy,gx],-5,5)).item()
                        cx_p=(gx+dx)*stride; cy_p=(gy+dy)*stride
                        bw_p=gw*stride; bh_p=gh*stride
                        all_boxes.append([cx_p-bw_p/2, cy_p-bh_p/2, cx_p+bw_p/2, cy_p+bh_p/2])
                        all_scores.append(final_conf[gy,gx].item())
                        all_cls.append(torch.argmax(cls_p[:,gy,gx]).item())
                    keep = torchvision.ops.nms(torch.tensor(all_boxes), torch.tensor(all_scores), 0.45)
                    for idx in keep:
                        x1,y1,x2,y2 = all_boxes[idx]
                        color = plt.cm.tab20(all_cls[idx] % 20)
                        rect = plt.Rectangle((x1,y1), x2-x1, y2-y1, fill=False,
                                              edgecolor=color, linewidth=1.5, alpha=min(1.0, all_scores[idx]*2))
                        ax.add_patch(rect)
                    ax.set_title(f"Pred Boxes (n={len(keep)})", fontsize=10)
                else:
                    ax.set_title("Pred Boxes (n=0)", fontsize=10)
            else:
                if disp_gt is not None:
                    ax.imshow(disp_gt[0].squeeze().cpu().numpy(), cmap='magma', vmin=0, vmax=vmax)
                    ax.set_title("GT Disparity", fontsize=10)
                else:
                    ax.set_title("No GT", fontsize=10)
            ax.axis('off')

        # Seg class legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=SEG_COLORS[i]/255.0, label=SEG_CLASS_NAMES[i])
                           for i in range(len(SEG_CLASS_NAMES))]
        fig.legend(handles=legend_elements, loc='lower center', ncol=6,
                   fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.01))

        plt.suptitle(f"V2.7 Dashboard — Step {step}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        writer.add_figure('Model_Progress/Dashboard', fig, global_step=step)
        plt.close(fig)
    if was_training: model.train()

print("✅ V2.7 Visualizer loaded")


## V2.7 Phase 1 Training
All heads simultaneously. 25 epochs. OneCycleLR.

In [ ]:
def hexapod_collate(batch):
    """Custom collate for mixed TartanAir/COCO batches."""
    result = {}
    result['left'] = torch.stack([b['left'] for b in batch])
    result['right'] = torch.stack([b['right'] for b in batch])
    result['teacher'] = torch.stack([b['teacher'] for b in batch])

    if batch[0]['disp'] is not None:
        result['disp'] = torch.stack([b['disp'] for b in batch])
    else:
        result['disp'] = None

    if batch[0]['seg'] is not None:
        result['seg'] = torch.stack([b['seg'] for b in batch])
    else:
        result['seg'] = None

    if batch[0]['normals'] is not None:
        result['normals'] = torch.stack([b['normals'] for b in batch])
        result['normals_valid'] = torch.stack([b['normals_valid'] for b in batch])
    else:
        result['normals'] = None
        result['normals_valid'] = None

    det_list = [b['det'] for b in batch]
    if det_list[0] is not None:
        result['det'] = det_list
    else:
        result['det'] = None

    return result

def infinite_loader(loader):
    while True:
        for batch in loader: yield batch

def get_grad_norm(params):
    total = 0.0
    for p in params:
        if p.grad is not None:
            total += p.grad.data.norm(2).item() ** 2
    return total ** 0.5

def save_checkpoint(model, criterion, optimizer, scheduler, scaler, epoch, loss, filename):
    path = os.path.join(CONFIG['save_dir'], filename)
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'epoch': epoch, 'loss': loss,
    }, path)
    print(f"  💾 Saved: {path} (loss={loss:.4f})")

print("✅ Helpers loaded")


In [ ]:
# =====================================================================
# 📦 DATA LOADING
# =====================================================================
print("Loading datasets...")

tartan_ds = V27Dataset(roots={'tartan': '../datasets/TartanAir'}, img_size=(CONFIG['img_height'], CONFIG['img_width']))
coco_ds = V27Dataset(roots={'coco': '../datasets/coco'}, img_size=(CONFIG['img_height'], CONFIG['img_width']))

tartan_loader = DataLoader(tartan_ds, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=CONFIG['num_workers'], pin_memory=False, persistent_workers=True,
    drop_last=True, collate_fn=hexapod_collate)
coco_loader = DataLoader(coco_ds, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=CONFIG['num_workers'], pin_memory=False, persistent_workers=True,
    drop_last=True, collate_fn=hexapod_collate)

print(f"TartanAir: {len(tartan_ds)} samples, {len(tartan_loader)} batches")
print(f"COCO: {len(coco_ds)} samples, {len(coco_loader)} batches")

# =====================================================================
# 🎓 TEACHERS (optional, for comparison only — not needed for training!)
# =====================================================================
print("\nLoading teachers (for visualization only)...")

# SegFormer teacher (optional — V2.7 doesn't need it for training)
try:
    from transformers import SegformerForSemanticSegmentation
    teacher_seg = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b2-finetuned-ade-512-512"
    ).to(DEVICE).eval()
    for p in teacher_seg.parameters(): p.requires_grad = False
    print("  SegFormer teacher loaded (for comparison only)")
except:
    teacher_seg = None
    print("  ⚠️ SegFormer not available — seg visualization will show model output only")

# YOLOv5 teacher (for COCO pseudo-label generation — already done offline)
# Not needed during training!
try:
    yolo_teacher = torch.hub.load('ultralytics/yolov5', 'yolov5l', pretrained=True).to(DEVICE).eval()
    for p in yolo_teacher.parameters(): p.requires_grad = False
    print("  YOLOv5l teacher loaded (for dashboard only)")
except:
    yolo_teacher = None
    print("  ⚠️ YOLOv5 teacher not available")

# =====================================================================
# 📸 DASHBOARD SETUP
# =====================================================================
print("\nSetting up dashboard...")

def get_batch_from_samples(dataset, target_filename):
    for i, sample in enumerate(dataset.samples):
        if target_filename in sample['l']:
            return hexapod_collate([dataset[i]])
    return hexapod_collate([dataset[0]])

TARTAN_TARGET = "/TartanAir/office2/Easy/P000/image_left/000336_left.png"
COCO_TARGET = "/coco/train2017/000000003145.jpg"

tartan_val = get_batch_from_samples(tartan_loader.dataset, TARTAN_TARGET)
coco_val = get_batch_from_samples(coco_loader.dataset, COCO_TARGET)

static_batches = {
    'tartan': tartan_val,
    'yolo': coco_val,
}

# Training state
global_step = 0
best_viz_loss = float('inf')
last_viz_step = 0
smoothed_loss = None
MIN_VIZ_STEPS = 100
MAX_VIZ_STEPS = 2000
LOSS_DROP_THRESH = 0.05

scaler = torch.cuda.amp.GradScaler()

# TensorBoard
writer = SummaryWriter(log_dir='./logs/ver_2-7_Phase-1')

print("✅ Dashboard ready!")


In [ ]:
# =====================================================================
# 🚀 V2.7 PHASE 1 TRAINING
# =====================================================================
os.makedirs(CONFIG['save_dir'], exist_ok=True)

criterion = V27HexapodLoss(CONFIG).to(DEVICE)

# Optimizer (5 groups + normals head)
base_lr = CONFIG['lr']
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(),      'lr': base_lr * 0.1, 'name': 'Backbone'},
    {'params': model.fpn_neck.parameters(),      'lr': base_lr * 1.0, 'name': 'FPN_Neck'},
    {'params': model.yolo_head.parameters(),     'lr': base_lr * 1.0, 'name': 'Yolo_Head'},
    {'params': model.stereo_head.parameters(),   'lr': base_lr * 1.0, 'name': 'Stereo_Head'},
    {'params': model.seg_head.parameters(),      'lr': base_lr * 1.0, 'name': 'Seg_Head'},
    {'params': model.normals_head.parameters(),  'lr': base_lr * 1.0, 'name': 'Normals_Head'},
], weight_decay=1e-4)

steps_per_epoch = len(coco_loader) // CONFIG['ACCUMULATION_STEPS']
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[base_lr*0.1, base_lr*1.0, base_lr*1.0, base_lr*1.0, base_lr*1.0, base_lr*1.0],
    epochs=CONFIG['num_epochs'],
    steps_per_epoch=steps_per_epoch,
    pct_start=0.1,
    div_factor=10.0,
    final_div_factor=1000.0,
)

# Gradient clipping
CLIP_NORMS = {
    'backbone': 3.0, 'fpn_neck': 5.0, 'yolo_head': 5.0,
    'stereo_head': 5.0, 'seg_head': 3.0, 'normals_head': 5.0,
}

print(f"Steps/epoch: {steps_per_epoch}")
print(f"Total steps: {steps_per_epoch * CONFIG['num_epochs']}")

best_loss = float('inf')

for epoch in range(CONFIG['num_epochs']):
    model.train()
    model.backbone.eval()  # BN locked
    running_loss = 0.0

    tartan_iter = iter(infinite_loader(tartan_loader))
    coco_iter = iter(infinite_loader(coco_loader))
    pbar = tqdm(range(steps_per_epoch), desc=f"V2.7 Ep {epoch+1}/{CONFIG['num_epochs']}")

    for step in pbar:
        global_step = epoch * steps_per_epoch + step
        total_step_loss = 0.0
        step_logs = {}

        optimizer.zero_grad(set_to_none=True)

        # --- TARTANAIR (Stereo + Normals + Seg) ---
        for _ in range(CONFIG['ACCUMULATION_STEPS']):
            batch = next(tartan_iter)
            l, r = batch['left'].to(DEVICE), batch['right'].to(DEVICE)
            targets = {
                'disp': batch['disp'].to(DEVICE),
                'normals': batch['normals'].to(DEVICE),
                'normals_valid': batch['normals_valid'].to(DEVICE),
                'seg': batch['seg'].to(DEVICE),
                'det': None,
            }
            with torch.cuda.amp.autocast():
                final_disp, seg_pred, det_pred, disp_s8, normals_pred = model(l, r)
            preds = (final_disp.float(), seg_pred.float(), None,
                     disp_s8.float() if disp_s8 is not None else None,
                     normals_pred.float())

            loss, logs = criterion(preds, targets, left_img=l, coarse_disp=preds[3])
            loss = loss / (2.0 * CONFIG['ACCUMULATION_STEPS'])
            scaler.scale(loss).backward()
            total_step_loss += loss.item()
            for k,v in logs.items():
                step_logs[k] = step_logs.get(k,0) + v/CONFIG['ACCUMULATION_STEPS']

        # --- COCO (YOLO only) ---
        for _ in range(CONFIG['ACCUMULATION_STEPS']):
            batch = next(coco_iter)
            l = batch['left'].to(DEVICE)
            coco_det = [t.to(DEVICE) for t in batch['det']]
            targets = {'disp': None, 'normals': None, 'seg': None, 'det': coco_det}

            with torch.cuda.amp.autocast():
                _, seg_pred, det_pred, _, normals_pred = model(l)
            det_pred = [d.float() for d in det_pred]
            preds = (None, None, det_pred, None, None)

            loss, logs = criterion(preds, targets, left_img=l)
            loss = loss / (2.0 * CONFIG['ACCUMULATION_STEPS'])
            scaler.scale(loss).backward()
            total_step_loss += loss.item()
            for k,v in logs.items():
                step_logs[k] = step_logs.get(k,0) + v/CONFIG['ACCUMULATION_STEPS']

        # --- STEP ---
        scaler.unscale_(optimizer)
        for tag, module in [('Backbone',model.backbone),('FPN',model.fpn_neck),
                            ('Stereo',model.stereo_head),('Seg',model.seg_head),
                            ('Yolo',model.yolo_head),('Normals',model.normals_head)]:
            step_logs[f'Grad_Norm_Pre/{tag}'] = get_grad_norm(module.parameters())
            clip_key = tag.lower() + ('_head' if tag not in ['Backbone','FPN'] else ('_neck' if tag=='FPN' else ''))
            torch.nn.utils.clip_grad_norm_(module.parameters(), max_norm=CLIP_NORMS.get(clip_key, 5.0))

        scaler.step(optimizer); scaler.update(); scheduler.step()
        running_loss += total_step_loss

        # Logging
        if step % 50 == 0:
            writer.add_scalar('Loss/Total_Weighted', total_step_loss, global_step)
            for i,pg in enumerate(optimizer.param_groups):
                writer.add_scalar(f'LR/{pg.get("name",f"G{i}")}', pg['lr'], global_step)
            for k,v in step_logs.items():
                if k.startswith('Grad_Norm'): writer.add_scalar(k, v, global_step)
                elif k not in ['yolo_num_targets']:
                    writer.add_scalar(f'Loss_Raw/{k}', v, global_step)

        # Track RAW loss sum for viz triggers (normalized total is ~constant)
        raw_loss = sum(step_logs.get(k, 0) for k in ['stereo', 'yolo', 'normals', 'seg_ce'])
        if smoothed_loss is None: smoothed_loss = raw_loss
        else: smoothed_loss = 0.9*smoothed_loss + 0.1*raw_loss

        tsl = global_step - last_viz_step
        sig_drop = True if best_viz_loss==float('inf') else (best_viz_loss-smoothed_loss)/(best_viz_loss+1e-8)>LOSS_DROP_THRESH
        if global_step==0 or (tsl>=MIN_VIZ_STEPS and sig_drop) or tsl>=MAX_VIZ_STEPS:
            print(f"\U0001f4f8 Snapshot step {global_step} | Loss: {smoothed_loss:.4f}")
            visualize_v27(step=global_step, writer=writer)
            writer.flush(); torch.cuda.empty_cache()
            model.train(); model.backbone.eval()
            best_viz_loss = min(best_viz_loss, smoothed_loss); last_viz_step = global_step

        pbar.set_postfix({
            'Total':f"{total_step_loss:.2f}",
            'Stereo':f"{step_logs.get('stereo',0):.2f}",
            'Seg':f"{step_logs.get('seg_ce',0):.2f}",
            'Yolo':f"{step_logs.get('yolo',0):.1f}",
            'Normals':f"{step_logs.get('normals',0):.3f}"})

    # Epoch-end snapshot + save
    epoch_loss = running_loss / steps_per_epoch
    visualize_v27(step=global_step, writer=writer)
    writer.flush(); torch.cuda.empty_cache()
    model.train(); model.backbone.eval()
    last_viz_step = global_step

    save_checkpoint(model=model, criterion=criterion, optimizer=optimizer,
                    scheduler=scheduler, scaler=scaler, epoch=epoch, loss=epoch_loss,
                    filename=f"checkpoint_v27_epoch_{epoch}.pth")
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        print(f"🌟 Best: {best_loss:.4f}!")
        save_checkpoint(model=model, criterion=criterion, optimizer=optimizer,
                        scheduler=scheduler, scaler=scaler, epoch=epoch, loss=epoch_loss,
                        filename="checkpoint_v27_best.pth")

print("✅ V2.7 Phase 1 Complete!")
